## What is the problem?
To build a machine learning model that takes as an input an audio recording of **10 seconds** of duration. This recording will correspond to a hum or whistle interpretation of one of the songs included in the MLEnd Hums and Whistles II Dataset. The output of the model will be the song label.


## Dataset
You can download them from:

https://github.com/thekmannn/MLEndHW_Sample/raw/main/MLEndHWII_Sample_400.zip

and

https://github.com/thekmannn/MLEndHW_Sample/raw/main/MLEndHWII_Sample_800.zip


# Using SVM, Logistic Regression and KNN to build an Audio Multiclassifier.

# 1 Problem formulation

The goal of this project is to build a machine learning model that can predict the song title from a 10 second audio clip of a hum or whistle. This is a multiclass audio classification problem as each song title corresponds to a different class.

There are many challenges with audio classification. First, the dataset I have chosen to use contains 400 samples. This is a relatively small dataset for 8 classes, and this might cause overfitting.

In addition to this, humming and whistling vary significantly amongst different participants, due to factors such as different pitch, tone, articulation and rhythm. Even audio clips created by the same person can vary significantly. This makes is difficult to recognise patterns for every song.

Real-world recording factors also increase the complexity. For example, background noise, mic, breaths, and overall recording quality can mask essential features in the audio clip.

Despite these challenges, the problem is interesting becuase it involves learning meaningful temporal and spectral representations from highly variable audio. Features such as pitch contours (F0), harmonic structure, spectral centroid, zero-crossing rate, and MFCCs capture important melodic and timbral information that encode the identity of a song (Faulkner & Rosen, 1999).

By modelling these representations, the classifier aims to learn underlying audio patterns across the different clips.



# 2 Methodology

This project follows the following machine learning pipeline: feature extraction, model training, and model evaluation. The objective is to correctly classify a 10 second audio clip of a humming or whistle segemnt into its song class.

In order to convert the raw audio clips into a structures numerical form that is suitable for machine learning, a set of temporal and spectral features was extracted from each clip. Pitch descriptors were included, such as the mean and standard deviation of the fundamental frequency (F0), along with the voiced-frame ratio, which altogether captures the meolodic contour of the audio.

By using pitch, the audio clips were then seperated according to whether it was identified to be a hum or a whistle. This was done to facilitate pattern recognition.

Signal power, which represents the overall amplitude and energy of the recording was calculated to reflect the characteristics of loudness. 13 MFCCs(Mel-Frequency Cepstral Coefficients) were calculated and summarised using their mean and standard deviation to represent the timbre. Additional spectral and harmonal features were extracted using chroma features (intensity of 12 pitch classes), spectral centroid (perceived brightness of sound), and zero-crossing rate (sharpness in the waveform). As a whole, these features capture the clips' melody, timbre and harmonic structure, which are essential for finding patterns between them.

Three machine learning models were trained for both the hum and whistle subsets: Support Vector Machine (SVM), Logictic Regression, and k-Nearest Neighbours (KNN). These models were selected as they work well with relatively small datasets, and over sufficient resistance to overfitting. SVM provides strong performance in high dimensional feature spaces, Logistic Regression offers a stable and interpretable linear baseline, and KNN provides a non-parametric approach which classifies based on neighbourly similarity. Training all three models on the same extracted features enables a direct comparison between them.

Model performance was evaluated using two metrics: classification accuracy and a confusion matrix. Accuracy gives an overall measure of how many audio clips were correctly classified, while the confusion matrix offers a more detailed view of class performance, showing which classes were frequently confused with others. These metrics were computed for each individual model as well as for the final ensemble model, which combined the predictions for these 3 models using voting.



# 3 Implemented ML prediction pipelines

Three machine learning pipelines were implemented. Each pipeline followed the same overal structure: convert audio files into numerical features, standardise the features, and train the classifier. A final ensemble pipeline combines the predictions.

The input to all pipelines is a list of audio file paths and their corresponding class labels. After transformation, the data structure consists of a feature matrix of recording and audio features. The outputs of each pipeline are class predictions, accuracy values and confusions matrices.

Three models were trained: SVM, Logistic Regression, and kNN, which were then combined for the ensemble.

Evaluation was performed using the accuracy scores and its confusion matrix.




## 3.1 Transformation stage

The transformationm stage converts the raw audio signals into numerical feature vectors suitable for machine learning classification. Several temporal and spectral features were extracted from each file. These included pitch measures such as the mean and standard deviation of the F0 and energy-related features such as signal power. To capture the spectral shape, 13 MFCCs were computed. Chroma features, zero-crossing rates, and the spectral centroid were also extracted.

After extraction, the features were standardised using normalisation to ensure zero meana nd unit variance. Standarisation was important as SVMs were highly sensitive to inconsistent feature scales, and both Logistic Regression and KNNs worked better with normalised features.

The output is therefore a fully standardised feature matrix paired with its vector label.



## 3.2 Model stage

Three machine learning models were implemented using the extracted and standardised features. The first model, a Support Vector Machine (SVM), constructs an optimal seperating margin between classes and performs well with small datasets. Its regualrisation parametre, c, and kernel choice were tuned to balance model flexibility, and prevent overfitting.

Logistic Regression was the next model, and this provided a linear decision boundary and estimated class proabilities. L2 regualarisation was applied to reduce variance and stabilise training given the relatively high dimensionality of the feature vector.

The last model used was k-Nearest Neihbours (KNN). This is non-parametric and instance-based. It assigns labels based on the most common class among the k closest training samples. As it relies on geometric similarity, it was useful to observe how well the feature spaces seperate the classes.

All three models were trained on the same training subset and were all evaluated using their accuracy score and confusion matrices.



## 3.3 Ensemble stage

To improve the reliability of the models' predictions, an ensemble approach merging the three models was used by voting.

The three classifiers first produce an independent prediction for every test instance, the final ensemble decision is the class selected by at least two of the three models.

This method was chosen as the models each work in different ways as they capture different parts of the feature distributions. Combining them reduces variance and also improves stability. This is especially prevalent in smaller datasets, where models might overfit or behave inconsistently. Voting also reduces the effect of incorrect predictions made by a single model.

The ensemble was also evaluted with an accuracy score and a confusion matrix, enabling direct comparison of whether combining decision boundaries improves classification.

# 4 Dataset

For this project, I chose to use the 400 samples datset, containing increments of hums and whistles. Each recording has data such as participant ID, song label, and interpretation type.

To prepare the data for machine learning, all audio cips were first loaded and inspected. Before feature extraction was carried out, the dataset was structured into a DataFrame containing file paths, participant data, and song labels. No modification was made to the actual audio, except amplitude normalisation to ensure consistent loudness.

Although the orginal dataset includes an interpreation label of whether it is a hum or whistle, an additional verification step was created in order to seperate the 400 samples into 2 subsets of whether it is a hum or whistle. This was conducted through pitch. Recordings with high F0 values were labelled as whistles, and recordings with low F0 values were classed as hums.

After gaining two subsets, the data now had to be seperated into train/test data. A 75:25 ratio was used for train:test.

As a result of this, 4 datasets were created: hum training set, hum testing set, whistle training set, whistle testing set.

Each split was performed deterministically, which ensured each audio appeared in a single subset. This helps the assumption that training sets contain both indepently and identically distributed (iid) samples.

One drawback to this however, is that a participant might appear in both the training and testing dataset, which creates similarities between sets.



# 5 Experiments and results


Loading python libraries and mounting google drive storage system

In [1]:
from google.colab import drive
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, sys, re, pickle, glob
import urllib.request
import zipfile
import math
import IPython.display as ipd
from tqdm import tqdm
import librosa
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix


drive.mount('/content/drive')

Mounted at /content/drive


Downloading the small sample of MLEnd Hums and Whistles Dataset (400 samples)

CW dataset:

https://github.com/thekmannn/MLEndHW_Sample/raw/main/MLEndHWII_Sample_400.zip

In [2]:
def download_url(url, save_path):
    with urllib.request.urlopen(url) as dl_file:
        total_size = int(dl_file.getheader('Content-Length', 0))
        block_size = 8192
        with open(save_path, 'wb') as out_file, tqdm(total=total_size, unit='B', unit_scale=True, desc="Downloading") as pbar:
            while True:
                data = dl_file.read(block_size)
                if not data:
                    break
                out_file.write(data)
                pbar.update(len(data))

Download MLEndHW_Sample 400

In [3]:
url  = "https://github.com/thekmannn/MLEndHW_Sample/raw/main/MLEndHWII_Sample_400.zip"
save_path = '/content/drive/MyDrive/Data/MLEndCW/MLEndHWII_Sample_400.zip'

download_url(url, save_path)

Downloading: 100%|██████████| 750M/750M [00:47<00:00, 15.8MB/s]


Running cell to confrim MLEndCW folder contains MLEndHW_Sample_400.zip

In [4]:
path = '/content/drive/MyDrive/Data/MLEndCW'
os.listdir(path)

['MLEndHWII_Sample_400.zip', 'sample']

Unzipping sample data and checking how many audio files we have

In [5]:
directory_to_extract_to = '/content/drive/MyDrive/Data/MLEndCW/sample/'
zip_path = '/content/drive/MyDrive/Data/MLEndCW/MLEndHWII_Sample_400.zip'
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(directory_to_extract_to)

sample_path = '/content/drive/MyDrive/Data/MLEndCW/sample/**/*.wav'
files = glob.glob(sample_path)
len(files)

400

Listening to some random audio files:

In [6]:
N = 5

for _ in range(N):
    idx = np.random.randint(len(files))
    file_path = files[idx]
    print("Playing:", os.path.basename(file_path))
    display(ipd.Audio(file_path))

Playing: S173_hum_2_Friend.wav


Playing: S107_whistle_2_Friend.wav


Playing: S96_hum_1_Married.wav


Playing: S137_hum_4_TryEverything.wav


Playing: S141_whistle_2_Happy.wav


Creating a datframe to show the song properties. From the DataFrame, we can see that the data contains file_id, participant, interpretation	number,song.

In [7]:
MLENDCW_table = []

for file in files:
    file_name = os.path.basename(file)
    parts = file_name.split('_')

    participant_ID = parts[0]
    interpretation_type = parts[1]
    interpretation_number = parts[2]
    song = parts[3].split('.')[0]

    MLENDCW_table.append([file_name, participant_ID, interpretation_type, interpretation_number, song])

MLENDCW_df = pd.DataFrame(
    MLENDCW_table,
    columns=['file_id', 'participant', 'interpretation', 'number', 'song']
).set_index('file_id')

MLENDCW_df.head(20)

,participant,interpretation,number,song
file_id,,,,
S100_whistle_2_RememberMe.wav,S100,whistle,2,RememberMe
S101_hum_2_Friend.wav,S101,hum,2,Friend
S101_hum_1_Married.wav,S101,hum,1,Married
S104_whistle_1_Married.wav,S104,whistle,1,Married
S100_whistle_2_Happy.wav,S100,whistle,2,Happy
S103_hum_3_RememberMe.wav,S103,hum,3,RememberMe
S100_hum_2_Married.wav,S100,hum,2,Married
S103_hum_1_Necessities.wav,S103,hum,1,Necessities
S104_whistle_2_Necessities.wav,S104,whistle,2,Necessities



Splitting up my dataset into 2 datasets: One hums and one whistles, before splitting into training / testing.

In [8]:
# extracting pitch in order to seperate hums and whistles
def get_pitch(x, fs, winLen=0.03):
    p = winLen * fs
    frame_length = int(2**int(p-1).bit_length()) + 256
    hop_length = frame_length // 2
    f0, voiced_flag, _ = librosa.pyin(
        y=x, fmin=80, fmax=450, sr=fs,
        frame_length=frame_length, hop_length=hop_length
    )

    pitch_mean = np.nanmean(f0) if np.mean(np.isnan(f0)) < 1 else 0
    pitch_std  = np.nanstd(f0) if np.mean(np.isnan(f0)) < 1 else 0
    voiced_fr  = np.mean(voiced_flag)
    return pitch_mean, pitch_std, voiced_fr

def extract_pitch_features(files, labels_df, scale_audio=True):
    X, y = [], []
    # loading audio, calc power, pitch features, and combining features
    for file in tqdm(files):
        fileID = os.path.basename(file)
        x, fs = librosa.load(file, sr=None)
        if scale_audio:
            x = x / np.max(np.abs(x))
        power = np.sum(x**2) / len(x)
        pitch_mean, pitch_std, voiced_fr = get_pitch(x, fs)

        # mfcc features
        # 13 coefficients
        mfcc = librosa.feature.mfcc(y=x, sr=fs, n_mfcc=13)
        mfcc_mean = np.mean(mfcc, axis=1)
        mfcc_std  = np.std(mfcc, axis=1)

        # chroma features
        chroma = librosa.feature.chroma_stft(y=x, sr=fs)
        chroma_mean = np.mean(chroma, axis=1)
        chroma_std  = np.std(chroma, axis=1)

        # spectral centroid features
        spec_centroid = librosa.feature.spectral_centroid(y=x, sr=fs)
        spec_centroid_mean = np.mean(spec_centroid)
        spec_centroid_std  = np.std(spec_centroid)

        # zero crossing rate features
        zcr = librosa.feature.zero_crossing_rate(x)
        zcr_mean = np.mean(zcr)
        zcr_std  = np.std(zcr)

        features = [power, pitch_mean, pitch_std, voiced_fr, *mfcc_mean, *mfcc_std, *chroma_mean, *chroma_std,
                    spec_centroid_mean, spec_centroid_std,
                    zcr_mean, zcr_std]
        X.append(features)
        yi = labels_df.loc[fileID]['song']
        y.append(yi)
    return np.array(X), np.array(y)

# seperating the files
hum_files = [f for f in files if MLENDCW_df.loc[os.path.basename(f)]['interpretation'] == 'hum']
whistle_files = [f for f in files if MLENDCW_df.loc[os.path.basename(f)]['interpretation'] == 'whistle']

print("Number of hum files:", len(hum_files))
print("Number of whistle files:", len(whistle_files))

Number of hum files: 201
Number of whistle files: 199


Now splitting my dataset into train/ test splits

In [9]:
from sklearn.model_selection import train_test_split
hum_train, hum_test = train_test_split(
    hum_files,
    test_size=0.25,
    random_state=42,
    stratify=[MLENDCW_df.loc[os.path.basename(f)]['song'] for f in hum_files]
)

whistle_train, whistle_test = train_test_split(
    whistle_files,
    test_size=0.25,
    random_state=42,
    stratify=[MLENDCW_df.loc[os.path.basename(f)]['song'] for f in whistle_files]
)

Extracting features for the train/test split

In [10]:
X_hum_train, y_hum_train = extract_pitch_features(hum_train, MLENDCW_df)
X_hum_test,  y_hum_test  = extract_pitch_features(hum_test, MLENDCW_df)

X_whistle_train, y_whistle_train = extract_pitch_features(whistle_train, MLENDCW_df)
X_whistle_test,  y_whistle_test  = extract_pitch_features(whistle_test, MLENDCW_df)

 60%|██████    | 90/150 [03:14<02:02,  2.04s/it]/usr/local/lib/python3.12/dist-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
100%|██████████| 50/50 [01:28<00:00,  1.77s/it]


Standardising features

In [11]:
scaler_hum = StandardScaler().fit(X_hum_train)
X_hum_train_scaled = scaler_hum.transform(X_hum_train)
X_hum_test_scaled  = scaler_hum.transform(X_hum_test)

scaler_whistle = StandardScaler().fit(X_whistle_train)
X_whistle_train_scaled = scaler_whistle.transform(X_whistle_train)
X_whistle_test_scaled  = scaler_whistle.transform(X_whistle_test)

Training models

1) SVM - Support Vector Machines

In [12]:
svm_hum = SVC(C=1, gamma='scale')
svm_hum.fit(X_hum_train_scaled, y_hum_train)
svm_wh = SVC(C=1, gamma='scale')
svm_wh.fit(X_whistle_train_scaled, y_whistle_train)

SVC(C=1)

2. Logistic Regression

In [13]:
log_reg_hum = LogisticRegression(max_iter=500)
log_reg_hum.fit(X_hum_train_scaled, y_hum_train)
log_reg_wh = LogisticRegression(max_iter=500)
log_reg_wh.fit(X_whistle_train_scaled, y_whistle_train)

LogisticRegression(max_iter=500)

3. KNN - K - Nearest Neighbours

In [14]:
knn_hum = KNeighborsClassifier(n_neighbors=5)
knn_hum.fit(X_hum_train_scaled, y_hum_train)
knn_wh = KNeighborsClassifier(n_neighbors=5)
knn_wh.fit(X_whistle_train_scaled, y_whistle_train)

KNeighborsClassifier()

Evaluating the 3 models

In [15]:
def evaluate_model(model, X_test, y_test, label="Model"):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    print(f"{label} Accuracy: {acc:.3f}")
    print(f"{label} Confusion Matrix:\n{cm}\n")

# Hums dataset
evaluate_model(svm_hum, X_hum_test_scaled, y_hum_test, "SVM (Hum)")
evaluate_model(log_reg_hum, X_hum_test_scaled, y_hum_test, "Logistic Regression (Hum)")
evaluate_model(knn_hum, X_hum_test_scaled, y_hum_test, "KNN (Hum)")

# Whistles dataset
evaluate_model(svm_wh, X_whistle_test_scaled, y_whistle_test, "SVM (Whistle)")
evaluate_model(log_reg_wh, X_whistle_test_scaled, y_whistle_test, "Logistic Regression (Whistle)")
evaluate_model(knn_wh, X_whistle_test_scaled, y_whistle_test, "KNN (Whistle)")

SVM (Hum) Accuracy: 0.235
SVM (Hum) Confusion Matrix:
[[1 2 1 1 0 1 0 1]
 [0 2 0 0 1 2 1 0]
 [1 0 1 3 0 0 0 1]
 [0 0 1 3 0 1 1 1]
 [0 3 0 2 1 0 0 0]
 [0 2 0 2 1 1 1 0]
 [0 0 0 0 4 0 2 0]
 [1 0 3 0 0 1 0 1]]

Logistic Regression (Hum) Accuracy: 0.216
Logistic Regression (Hum) Confusion Matrix:
[[0 1 3 0 0 1 0 2]
 [0 1 0 0 2 2 1 0]
 [0 0 5 1 0 0 0 0]
 [0 0 0 0 3 2 1 1]
 [0 1 0 1 2 0 1 1]
 [1 1 0 1 1 2 1 0]
 [1 1 0 0 4 0 0 0]
 [1 0 0 1 1 2 0 1]]

KNN (Hum) Accuracy: 0.196
KNN (Hum) Confusion Matrix:
[[0 1 0 1 3 0 2 0]
 [1 3 0 0 0 1 0 1]
 [1 0 2 3 0 0 0 0]
 [4 1 1 1 0 0 0 0]
 [2 2 0 0 1 0 0 1]
 [2 2 0 1 1 0 1 0]
 [0 1 0 0 2 0 3 0]
 [3 0 0 1 2 0 0 0]]

SVM (Whistle) Accuracy: 0.120
SVM (Whistle) Confusion Matrix:
[[1 1 2 0 0 1 1 0]
 [0 0 1 0 2 1 1 1]
 [0 1 2 0 0 1 1 1]
 [0 0 4 0 1 1 0 0]
 [0 1 2 0 0 1 0 2]
 [2 0 0 0 2 2 0 0]
 [1 0 1 1 1 2 1 0]
 [0 2 3 0 1 1 0 0]]

Logistic Regression (Whistle) Accuracy: 0.180
Logistic Regression (Whistle) Confusion Matrix:
[[1 1 2 0 0 1 1 0]
 [0 0 1 0 2 2 1

The accuracy score are seen to be quite low, and the confusion matrices show that there is a lot of discrepency between all the songs. An ensemble will next be used in attempt to improve accuracy.

Ensemble for the 3 trained models

In [16]:
from collections import Counter
from sklearn.metrics import accuracy_score, confusion_matrix
print("SVM Hum predictions:", svm_hum.predict(X_hum_test)[:10])
print("LogReg Hum predictions:", log_reg_hum.predict(X_hum_test)[:10])
print("KNN Hum predictions:", knn_hum.predict(X_hum_test)[:10])

# in order to vote for the best model
def ensemble_predict(models, X):
    # predictions from all models
    all_predictions = [model.predict(X) for model in models]
    # list arrays
    all_predictions = np.array(all_predictions)

    ensemble_preds = []
    for i in range(all_predictions.shape[1]):
        sample_preds = all_predictions[:, i]
        most_common = Counter(sample_preds).most_common(1)[0][0]
        ensemble_preds.append(most_common)

    return np.array(ensemble_preds)


# ensemble for hums dataset
hum_models = [svm_hum, log_reg_hum, knn_hum]
y_hum_pred = ensemble_predict(hum_models, X_hum_test)

hum_acc = accuracy_score(y_hum_test, y_hum_pred)
hum_cm  = confusion_matrix(y_hum_test, y_hum_pred, labels=np.unique(y_hum_test))

print("Ensemble Hum Accuracy:", hum_acc)
print("Ensemble Hum Confusion Matrix:\n", hum_cm)

# ensemble for whistles dataset
whistle_models = [svm_wh, log_reg_wh, knn_wh]
y_whistle_pred = ensemble_predict(whistle_models, X_whistle_test)

whistle_acc = accuracy_score(y_whistle_test, y_whistle_pred)
whistle_cm  = confusion_matrix(y_whistle_test, y_whistle_pred, labels=np.unique(y_whistle_test))

print("Ensemble Whistle Accuracy:", whistle_acc)
print("Ensemble Whistle Confusion Matrix:\n", whistle_cm)

SVM Hum predictions: ['Happy' 'Happy' 'Happy' 'Happy' 'Happy' 'Happy' 'Happy' 'Happy' 'Happy'
 'Happy']
LogReg Hum predictions: ['Friend' 'Friend' 'Friend' 'Friend' 'Friend' 'Friend' 'Friend' 'Friend'
 'Friend' 'Friend']
KNN Hum predictions: ['Feeling' 'Feeling' 'Friend' 'Feeling' 'Feeling' 'Friend' 'Friend'
 'Friend' 'Feeling' 'Friend']
Ensemble Hum Accuracy: 0.13725490196078433
Ensemble Hum Confusion Matrix:
 [[0 5 2 0 0 0 0 0]
 [0 3 3 0 0 0 0 0]
 [0 2 4 0 0 0 0 0]
 [0 4 3 0 0 0 0 0]
 [0 3 3 0 0 0 0 0]
 [0 3 4 0 0 0 0 0]
 [0 4 2 0 0 0 0 0]
 [0 3 3 0 0 0 0 0]]
Ensemble Whistle Accuracy: 0.12
Ensemble Whistle Confusion Matrix:
 [[0 0 6 0 0 0 0 0]
 [0 0 6 0 0 0 0 0]
 [0 0 6 0 0 0 0 0]
 [0 0 6 0 0 0 0 0]
 [0 0 5 1 0 0 0 0]
 [0 0 6 0 0 0 0 0]
 [0 0 7 0 0 0 0 0]
 [0 0 7 0 0 0 0 0]]


# 7 Conclusions

Overall, this project investigated the task of the classification of audio clips into eight classes using 3 machine learning models. The pipeline included preprocessing the audio, extracting features, seperating the hums and whistles into two subsets, training 3 classifiers, and performing an ensemble.

Across all the experiments, the model performance was relatively low. For the humming dataset, the accuracies ranged from 19-23% for individual models, with SVM performing best. For whistling, results were even lower, ranging from 10-18%. The ensemble also did not improve performance, achieving 13.7% accuracy for hums and 12% for whistles. The confusion matrices showed a substantial amount of misclassification across all categories, indicating that the models struggled.

The results show that pattern recognition from humming and whistling is quite challenges, especially with a small dataset of 400 samples, a great class imabalance, MFCC features and classical machine learning models. Humming or whistling a song compared to singing it also contains far less emotion and depth to the audio.

Despite low performance, the project did demonstrate a reasonable pipeline for audio classification, with the application of preprocessing, feature extraction, training and testing models, and attempting to see patterns between audio.

# 7.1 Future Improvements

One improvement that can be made is to use the 800 samples to see if feeding the models with more data, helps with classification.

Another improvement would be to extract more features that capture emotional depth such as mel-spectograms. Enrichening the feature set gives the models more patterns to work with.

The way the dataset was split can also be improved. Instead of a random 75:25 ratio split, k-fold cross validation could be performed in order to have more reliable comparisons across models.

The hyperparametres of the models should also be used. For example in SVM, the c and gamma can be adjusted, for KNN, different k values and distance metrics can be used, and with Logistic Regression, regularisation strengths.

Other classification models can be used, such as a Naive Bayes classfier, the use of neural networks, and random CNNs.

The ensemble strategy that is currently used is fairly simple, this can be greatly improved by the use of weighted voting and averaging.

Whilst the accuracies were low, this emphasises the complexity of pattern recognition in audio, and the limitations the datasets and models used had. As a result, this gives us a strong foundation on how to improve the classification for future work.

# 8 References




Faulkner, A. and Rosen, S. (1999) Contributions of temporal encodings of voicing, voicelessness, fundamental frequency, and amplitude variation to audio-visual and auditory speech perception. *J. Acoust. Soc. Am.*, 106, pp. 2063–2073. doi:10.1121/1.427951.

Mierswa, I. and Morik, K. (2005) Automatic Feature Extraction for Classifying Audio Data. Machine Learning, 58, pp. 127–149.

The following libraries and tools were used in this project:

Python

Google Colaboratory

NumPy

Pandas

Matplotlib

Scikit-Learn

Librosa

IPython

Standard built-in python modules:

os, sys, re, pickle, glob, urrlib.request, zipfile, and math